# Evaluador de calidad de datos abiertos
**Proyecto:** Desarrollo de un prototipo de software para la evaluación automatizada de la calidad de datos abiertos: completitud, unicidad y consistencia
**Autora:** Ana Yibe Zamora Rivera · Semillero Sedetes · UNIMINUTO Virtual
**Versión:** Parte 1 de 4 (v1.1) — Preparación y obtención de los datos

---
### Cómo usar este cuaderno
1. Ejecute los pasos **en orden**, de arriba hacia abajo, con el botón ▶ que aparece a la izquierda de cada celda.
2. Cuando una celda haga una pregunta, escriba la respuesta en el recuadro y presione **Enter**. Si solo presiona Enter, se usa el valor sugerido entre corchetes.
3. No es necesario modificar el código. Las celdas marcadas con 🔧 contienen funciones y pueden permanecer cerradas.

> Las demás partes (validación, reglas de calidad y reporte) se agregarán en las siguientes versiones.

## Paso 1. Preparar el entorno
Importa las bibliotecas que usa el prototipo. En Google Colab ya vienen instaladas.

In [1]:
#@title ▶ Paso 1. Preparar el entorno
import importlib, subprocess, sys

for biblioteca in ["pandas", "requests"]:
    try:
        importlib.import_module(biblioteca)
    except ImportError:
        print(f"Instalando {biblioteca}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", biblioteca], check=True)

import pandas as pd
import requests
print(f"✅ Entorno listo · Python {sys.version.split()[0]} · pandas {pd.__version__}")

✅ Entorno listo · Python 3.13.18 · pandas 2.2.3


## Paso 2. Conectar Google Drive
Conecta su Google Drive para guardar la carpeta del proyecto, los parámetros y el registro de cada ejecución.
Cuando aparezca la ventana de permisos, elija su cuenta de Google y presione **Permitir**.

In [2]:
#@title ▶ Paso 2. Conectar Google Drive
from pathlib import Path

NOMBRE_CARPETA = "EvaluadorCalidadDatosAbiertos"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_PROYECTO = Path("/content/drive/MyDrive") / NOMBRE_CARPETA
    CARPETA_TEMPORAL = Path("/content/temporal")
except ImportError:
    # Fuera de Colab (solo para pruebas): se trabaja en la carpeta actual.
    CARPETA_PROYECTO = Path.cwd() / NOMBRE_CARPETA
    CARPETA_TEMPORAL = Path.cwd() / "temporal"

CARPETA_PROYECTO.mkdir(parents=True, exist_ok=True)
CARPETA_TEMPORAL.mkdir(parents=True, exist_ok=True)
print(f"✅ Carpeta del proyecto: {CARPETA_PROYECTO}")

Mounted at /content/drive
✅ Carpeta del proyecto: /content/drive/MyDrive/EvaluadorCalidadDatosAbiertos


## Paso 3. Cargar las funciones
Carga las funciones de búsqueda, descarga, carga de archivos, parámetros y auditoría, y lee el archivo `parametros.json`.
Si es la primera vez, el archivo se crea con los valores por defecto.

In [3]:
#@title 🔧 Funciones de la Parte 1 (no es necesario modificar esta celda) { display-mode: "form" }
# ---------------------------------------------------------------------------
# Capa de acceso a datos y gestor de ejecuciones (Fase 3, secciones 4.1 y 6.4)
#   - parametros: lectura y escritura de parametros.json (RF-11)
#   - ejecuciones y auditoria: carpeta por ejecución y registro de eventos
#   - busqueda: consulta del catálogo de datos.gov.co y descarga del CSV (RF-12)
#   - carga: archivo CSV propio desde el equipo del usuario (RF-01)
# ---------------------------------------------------------------------------
import csv
import hashlib
import io
import json
import re
import unicodedata
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

URL_CATALOGO = "https://www.datos.gov.co/api/catalog/v1"
DOMINIO_PORTAL = "www.datos.gov.co"
URL_RECURSO = "https://www.datos.gov.co/resource/{id}.csv"
URL_CONTEO = "https://www.datos.gov.co/resource/{id}.json"
URL_FICHA = "https://www.datos.gov.co/d/{id}"
PATRON_ID = re.compile(r"^[a-z0-9]{4}-[a-z0-9]{4}$")

PARAMETROS_POR_DEFECTO = {
    "version_parametros": "1.0",
    "marcadores_vacio": ["", "N/A", "NA", "NULL", "-"],
    "formatos_fecha": ["%Y-%m-%d", "%d/%m/%Y", "%Y-%m-%d %H:%M:%S"],
    "max_resultados_busqueda": 20,
    "max_registros_descarga": 100000,
    "registros_por_pagina": 50000,
    "tamano_max_mb": 50,
    "max_ejemplos_por_regla": 10,
    "segundos_espera_red": 60,
}

ZONA_HORARIA = ZoneInfo("America/Bogota")  # Colab trabaja en hora UTC; se usa la hora de Colombia

SUBCARPETAS = ["01_configuracion", "02_busqueda", "03_fuente", "04_evaluacion", "05_reporte", "99_auditoria"]


# ----------------------------- utilidades ---------------------------------
def ahora():
    return datetime.now(ZONA_HORARIA).strftime("%Y-%m-%d %H:%M:%S")


def guardar_json(ruta, contenido):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    temporal = ruta.with_suffix(ruta.suffix + ".tmp")
    temporal.write_text(json.dumps(contenido, ensure_ascii=False, indent=2), encoding="utf-8")
    temporal.replace(ruta)


def leer_json(ruta, valor_si_no_existe=None):
    ruta = Path(ruta)
    if not ruta.exists():
        return valor_si_no_existe
    return json.loads(ruta.read_text(encoding="utf-8"))


def texto_para_carpeta(texto, largo=30):
    sin_tildes = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    limpio = re.sub(r"[^a-zA-Z0-9]+", "_", sin_tildes).strip("_").lower()
    return (limpio or "evaluacion")[:largo]


def preguntar(mensaje, valor_sugerido=""):
    """Pregunta al usuario. Si solo presiona Enter, se usa el valor sugerido."""
    ayuda = f" [Enter = {valor_sugerido}]" if valor_sugerido != "" else ""
    try:
        respuesta = input(f"{mensaje}{ayuda}: ").strip()
    except EOFError:
        respuesta = ""
    return respuesta or str(valor_sugerido)


def huella_sha256(ruta):
    calculadora = hashlib.sha256()
    with open(ruta, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            calculadora.update(bloque)
    return calculadora.hexdigest()


# ----------------------------- parámetros ---------------------------------
def cargar_parametros(carpeta_proyecto):
    """Lee parametros.json; si falta o tiene valores inválidos, usa los valores por defecto."""
    ruta = Path(carpeta_proyecto) / "parametros.json"
    guardados = leer_json(ruta, {}) or {}
    parametros = dict(PARAMETROS_POR_DEFECTO)
    for clave, valor in guardados.items():
        if clave not in PARAMETROS_POR_DEFECTO:
            continue
        esperado = type(PARAMETROS_POR_DEFECTO[clave])
        if isinstance(valor, esperado) and not (isinstance(valor, int) and valor <= 0):
            parametros[clave] = valor
        else:
            print(f"⚠️ El parámetro «{clave}» no es válido; se usa el valor por defecto.")
    if not ruta.exists():
        guardar_json(ruta, parametros)
        print("Se creó parametros.json con los valores por defecto.")
    return parametros


# ------------------------ ejecuciones y auditoría -------------------------
def crear_ejecucion(carpeta_proyecto, nombre):
    marca = datetime.now(ZONA_HORARIA).strftime("%Y%m%d-%H%M%S")
    identificador = f"{marca}_{texto_para_carpeta(nombre)}"
    carpeta = Path(carpeta_proyecto) / "ejecuciones" / identificador
    for sub in SUBCARPETAS:
        (carpeta / sub).mkdir(parents=True, exist_ok=True)
    ejecucion = {
        "id_ejecucion": identificador,
        "carpeta": str(carpeta),
        "fecha_hora": ahora(),
        "nombre": nombre,
        "origen": None,
        "estado": "creada",
    }
    guardar_json(carpeta / "01_configuracion" / "configuracion.json", ejecucion)
    guardar_json(carpeta / "99_auditoria" / "auditoria.json", [])
    registrar_evento(ejecucion, "crear_ejecucion", "ok", f"Carpeta {identificador}")
    return ejecucion


def actualizar_ejecucion(ejecucion, **cambios):
    ejecucion.update(cambios)
    guardar_json(Path(ejecucion["carpeta"]) / "01_configuracion" / "configuracion.json", ejecucion)


def registrar_evento(ejecucion, paso, resultado, detalle=""):
    ruta = Path(ejecucion["carpeta"]) / "99_auditoria" / "auditoria.json"
    eventos = leer_json(ruta, []) or []
    eventos.append({"momento": ahora(), "paso": paso, "resultado": resultado, "detalle": str(detalle)[:500]})
    guardar_json(ruta, eventos)


# ------------------------ búsqueda en datos.gov.co ------------------------
def buscar_conjuntos(tema, parametros):
    """Consulta el catálogo de datos.gov.co y devuelve una tabla con los conjuntos encontrados."""
    respuesta = requests.get(
        URL_CATALOGO,
        params={
            "q": tema,
            "domains": DOMINIO_PORTAL,
            "search_context": DOMINIO_PORTAL,
            "only": "dataset",
            "limit": parametros["max_resultados_busqueda"],
        },
        timeout=parametros["segundos_espera_red"],
    )
    respuesta.raise_for_status()
    filas = []
    for item in respuesta.json().get("results", []):
        recurso = item.get("resource", {})
        identificador = recurso.get("id", "")
        if recurso.get("type") != "dataset" or not PATRON_ID.match(identificador):
            continue
        filas.append({
            "id_conjunto": identificador,
            "titulo": recurso.get("name", ""),
            "entidad": recurso.get("attribution") or "Sin entidad registrada",
            "actualizado": (recurso.get("data_updated_at") or recurso.get("updatedAt") or "")[:10],
            "url_origen": URL_FICHA.format(id=identificador),
        })
    tabla = pd.DataFrame(filas, columns=["id_conjunto", "titulo", "entidad", "actualizado", "url_origen"])
    tabla.index = range(1, len(tabla) + 1)
    return tabla


def contar_registros(id_conjunto, parametros):
    try:
        respuesta = requests.get(URL_CONTEO.format(id=id_conjunto), params={"$select": "count(*) as total"},
                                 timeout=parametros["segundos_espera_red"])
        respuesta.raise_for_status()
        return int(respuesta.json()[0]["total"])
    except Exception:
        return None  # el conteo es informativo; si falla, la descarga continúa


def descargar_conjunto(id_conjunto, destino, parametros):
    """Descarga el conjunto por páginas hasta el máximo de registros y lo guarda como un único CSV."""
    total = contar_registros(id_conjunto, parametros)
    limite = parametros["max_registros_descarga"] if total is None else min(total, parametros["max_registros_descarga"])
    tamano_max = parametros["tamano_max_mb"] * 1024 * 1024
    encabezado, escritos, bytes_escritos = None, 0, 0
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    with destino.open("w", encoding="utf-8", newline="") as salida:
        escritor = csv.writer(salida)
        while escritos < limite:
            cantidad = min(parametros["registros_por_pagina"], limite - escritos)
            respuesta = requests.get(URL_RECURSO.format(id=id_conjunto),
                                     params={"$limit": cantidad, "$offset": escritos, "$order": ":id"},
                                     timeout=parametros["segundos_espera_red"])
            respuesta.raise_for_status()
            bytes_escritos += len(respuesta.content)
            if bytes_escritos > tamano_max:
                raise ValueError(f"El conjunto supera el tamaño máximo de {parametros['tamano_max_mb']} MB.")
            lector = csv.reader(io.StringIO(respuesta.content.decode("utf-8-sig"), newline=""))
            columnas = next(lector, None)
            registros = list(lector)
            if columnas is None or not registros:
                break
            if encabezado is None:
                encabezado = columnas
                escritor.writerow(encabezado)
            elif columnas != encabezado:
                raise ValueError("Las columnas cambiaron entre páginas; la descarga se detuvo.")
            escritor.writerows(registros)
            escritos += len(registros)
            print(f"   {escritos:,} registros descargados" + (f" de {limite:,}" if limite else ""))
            if len(registros) < cantidad:
                break
    if not escritos:
        destino.unlink(missing_ok=True)
        raise ValueError("El conjunto de datos no tiene registros para descargar.")
    cobertura = "completa" if total is not None and escritos >= total else (
        "parcial por límite" if total is not None else "sin conteo del portal")
    return {"registros_descargados": escritos, "registros_en_portal": total, "cobertura": cobertura}


# --------------------------- carga de archivo propio ----------------------
def cargar_archivo_propio(carpeta_temporal, parametros):
    """Pide al usuario un CSV desde su equipo (en Colab abre el selector de archivos)."""
    carpeta_temporal = Path(carpeta_temporal)
    carpeta_temporal.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import files
    except ImportError:
        ruta = Path(preguntar("Escriba la ruta del archivo CSV"))
        if not ruta.exists():
            raise FileNotFoundError(f"No existe el archivo {ruta}")
        nombre, contenido = ruta.name, ruta.read_bytes()
    else:
        print("Seleccione el archivo CSV en la ventana que se abre...")
        subidos = files.upload()
        if not subidos:
            raise ValueError("No se seleccionó ningún archivo.")
        nombre, contenido = next(iter(subidos.items()))
    if not nombre.lower().endswith(".csv"):
        raise ValueError("El archivo debe tener extensión .csv.")
    if len(contenido) > parametros["tamano_max_mb"] * 1024 * 1024:
        raise ValueError(f"El archivo supera el tamaño máximo de {parametros['tamano_max_mb']} MB.")
    destino = carpeta_temporal / texto_para_carpeta(Path(nombre).stem, 60)
    destino = destino.with_suffix(".csv")
    destino.write_bytes(contenido)
    return destino, nombre


print("✅ Funciones de la Parte 1 cargadas.")


parametros = cargar_parametros(CARPETA_PROYECTO)
print("Parámetros en uso:")
for clave, valor in parametros.items():
    print(f"   {clave}: {valor}")

✅ Funciones de la Parte 1 cargadas.
Parámetros en uso:
   version_parametros: 1.0
   marcadores_vacio: ['', 'N/A', 'NA', 'NULL', '-']
   formatos_fecha: ['%Y-%m-%d', '%d/%m/%Y', '%Y-%m-%d %H:%M:%S']
   max_resultados_busqueda: 20
   max_registros_descarga: 100000
   registros_por_pagina: 50000
   tamano_max_mb: 50
   max_ejemplos_por_regla: 10
   segundos_espera_red: 60


## Paso 4. Crear la ejecución
Cada uso del prototipo es una **ejecución**. Se crea una carpeta con la fecha, la hora y el nombre que usted escriba, donde quedarán la configuración, el reporte y la auditoría.

In [4]:
#@title ▶ Paso 4. Crear la ejecución
nombre_ejecucion = preguntar("Escriba un nombre corto para esta ejecución (por ejemplo, el tema)", "evaluacion")
ejecucion = crear_ejecucion(CARPETA_PROYECTO, nombre_ejecucion)
print(f"✅ Ejecución creada: {ejecucion['id_ejecucion']}")
print(f"   Carpeta: {ejecucion['carpeta']}")

Escriba un nombre corto para esta ejecución (por ejemplo, el tema) [Enter = evaluacion]: educacion
✅ Ejecución creada: 20261115-214759_educacion
   Carpeta: /content/drive/MyDrive/EvaluadorCalidadDatosAbiertos/ejecuciones/20261115-214759_educacion


## Paso 5A. Buscar o cargar los datos
Elija de dónde vienen los datos:
- **1. Buscar en datos.gov.co:** escribe un tema y el prototipo muestra una lista numerada de los conjuntos de datos encontrados. La elección se hace en el **Paso 5B**.
- **2. Cargar un archivo propio:** selecciona un archivo CSV de su computador. En este caso el Paso 5B no es necesario.

El archivo de datos se guarda solo en el almacenamiento temporal de Colab; en Drive quedan únicamente los datos de la fuente y la auditoría.

In [5]:
#@title ▶ Paso 5A. Buscar o cargar los datos
from pathlib import Path

ruta_datos = None
fuente = None
resultados_busqueda = None
tema_busqueda = None

def mostrar_vista(ruta, datos_fuente):
    guardar_json(Path(ejecucion["carpeta"]) / "03_fuente" / "fuente.json", datos_fuente)
    vista = pd.read_csv(ruta, dtype=str, nrows=5, sep=None, engine="python")
    print(f"\n✅ Datos listos para la validación ({datos_fuente['origen']}).")
    print("Primeros registros (la validación completa se hace en la Parte 2):")
    display(vista)

def obtener_desde_equipo():
    destino, nombre_original = cargar_archivo_propio(CARPETA_TEMPORAL, parametros)
    datos_fuente = {"origen": "archivo propio", "nombre_original": nombre_original,
                    "archivo_temporal": str(destino), "huella_sha256": huella_sha256(destino),
                    "tamano_bytes": destino.stat().st_size, "fecha_carga": ahora()}
    actualizar_ejecucion(ejecucion, origen="archivo propio", estado="datos obtenidos")
    registrar_evento(ejecucion, "carga", "ok", nombre_original)
    return destino, datos_fuente

def mostrar_lista(tabla):
    print(f"\nSe encontraron {len(tabla)} conjuntos de datos:\n")
    for numero, fila in tabla.iterrows():
        titulo = fila["titulo"] if len(fila["titulo"]) <= 75 else fila["titulo"][:72] + "..."
        print(f"{numero:>3}. {titulo}")
        print(f"      {fila['entidad']} · actualizado {fila['actualizado'] or 'sin fecha'}")

opcion = preguntar("¿De dónde vienen los datos? 1 = buscar en datos.gov.co, 2 = cargar un archivo propio", 1)
try:
    if opcion == "2":
        ruta_datos, fuente = obtener_desde_equipo()
        mostrar_vista(ruta_datos, fuente)
        print("\nComo cargó un archivo propio, no necesita el Paso 5B.")
    else:
        tema_busqueda = preguntar("¿Qué tema desea buscar en datos.gov.co?", ejecucion["nombre"])
        print(f"Buscando «{tema_busqueda}» en datos.gov.co...")
        resultados_busqueda = buscar_conjuntos(tema_busqueda, parametros)
        guardar_json(Path(ejecucion["carpeta"]) / "02_busqueda" / "resultados.json",
                     {"tema": tema_busqueda, "fecha_hora": ahora(),
                      "resultados": resultados_busqueda.to_dict(orient="records")})
        registrar_evento(ejecucion, "busqueda", "ok", f"Tema «{tema_busqueda}»: {len(resultados_busqueda)} conjuntos")
        if resultados_busqueda.empty:
            print("No se encontraron conjuntos de datos con ese tema. Ejecute de nuevo este paso con otro tema.")
        else:
            mostrar_lista(resultados_busqueda)
            print("\n👉 Revise la lista y ejecute el Paso 5B para escribir el número del conjunto que desea evaluar.")
except Exception as error:
    registrar_evento(ejecucion, "obtener_datos", "error", error)
    print(f"❌ No se pudieron obtener los datos: {error}")
    if opcion != "2":
        print("Puede intentar con otro tema o elegir la opción 2 para cargar un archivo propio.")

¿De dónde vienen los datos? 1 = buscar en datos.gov.co, 2 = cargar un archivo propio [Enter = 1]: 
¿Qué tema desea buscar en datos.gov.co? [Enter = educacion]: 
Buscando «educacion» en datos.gov.co...

Se encontraron 20 conjuntos de datos:

  1. EDUCACION AMBIENTAL MUNICIPIO DE CHIA
      Alcaldía de Chía, Cundinamarca · actualizado 2023-08-09
  2. Resumen de Estrategias	Secretaría Educacion - Santander
      Gobernación de Santander, Santander · actualizado 2026-10-16
  3. MEN_MATRICULA_ESTADISTICA_ES
      Ministerio de Educación Nacional - MinEducación, Bogotá D.C. · actualizado 2025-10-28
  4. Instituciones Educativas Municipio de Envigado
      Alcaldía de Envigado, Antioquia · actualizado 2024-08-30
  5. Listado de Instituciones Educativas Oficiales y No Oficiales del Departa...
      Gobernación del Atlántico, Atlántico · actualizado 2024-08-21
  6. NÚMERO DE ALUMNOS MATRICULADOS POR MUNICIPIO Y POR GRADO EN EL DEPARTAME...
      Gobernación del Cesar, Cesar · actualizado 2020-0

## Paso 5B. Elegir y descargar el conjunto
Escriba el número del conjunto de datos de la lista del Paso 5A. El prototipo lo descarga desde datos.gov.co y muestra los primeros registros.

In [6]:
#@title ▶ Paso 5B. Elegir y descargar el conjunto
def pedir_numero(cantidad, intentos=5):
    for _ in range(intentos):
        respuesta = preguntar(f"Escriba el número del conjunto que desea evaluar (de 1 a {cantidad})")
        if respuesta.isdigit() and 1 <= int(respuesta) <= cantidad:
            return int(respuesta)
        print(f"   «{respuesta}» no es un número de la lista. Intente de nuevo.")
    raise ValueError("No se eligió un número válido.")

if fuente is not None:
    print("Los datos ya se obtuvieron en el Paso 5A. Puede continuar con la Parte 2.")
elif resultados_busqueda is None or resultados_busqueda.empty:
    print("Primero ejecute el Paso 5A y busque un tema que tenga resultados.")
else:
    try:
        numero = pedir_numero(len(resultados_busqueda))
        elegido = resultados_busqueda.loc[numero].to_dict()
        print(f"Descargando «{elegido['titulo']}»...")
        destino = CARPETA_TEMPORAL / f"{elegido['id_conjunto']}.csv"
        resumen = descargar_conjunto(elegido["id_conjunto"], destino, parametros)
        fuente = {**elegido, **resumen, "origen": "datos.gov.co", "tema_busqueda": tema_busqueda,
                  "archivo_temporal": str(destino), "huella_sha256": huella_sha256(destino),
                  "fecha_descarga": ahora()}
        ruta_datos = destino
        actualizar_ejecucion(ejecucion, origen="datos.gov.co", tema_busqueda=tema_busqueda,
                             id_conjunto=elegido["id_conjunto"], estado="datos obtenidos")
        registrar_evento(ejecucion, "descarga", "ok",
                         f"{elegido['id_conjunto']}: {resumen['registros_descargados']} registros ({resumen['cobertura']})")
        if resumen["cobertura"] == "parcial por límite":
            print(f"⚠️ El portal tiene {resumen['registros_en_portal']:,} registros; se descargaron los primeros "
                  f"{resumen['registros_descargados']:,} por el límite de los parámetros.")
        mostrar_vista(ruta_datos, fuente)
    except Exception as error:
        registrar_evento(ejecucion, "descarga", "error", error)
        print(f"❌ No se pudo descargar el conjunto: {error}")
        print("Puede ejecutar de nuevo este paso y elegir otro número.")

Escriba el número del conjunto que desea evaluar (de 1 a 20): 15
Descargando «MEN_INSTITUCIONES EDUCACIÓN SUPERIOR»...
   361 registros descargados de 361

✅ Datos listos para la validación (datos.gov.co).
Primeros registros (la validación completa se hace en la Parte 2):


,c_digo_instituci_n,nombre_instituci_n,n_mero_identificaci_n,principal_seccional,naturaleza_jur_dica,sector,car_cter_acad_mico,cod_departamento,departamento_domicilio,cod_municipio,...,tel_fono_domicilio,norma_de_creaci_n,fecha_norma_de_creaci_n,acreditada_alta_calidad,fecha_acreditaci_n,resoluci_n_de_la_acreditaci,vigencia_de_la_acreditaci,estado,fecha_actualizacion,p_gina_web
0,1123,UNIVERSIDAD POPULAR DEL CESAR,892.300.285-6,Seccional,Nacional,Oficial,Universidad,Cesar,Cesar,NaN,...,5654900,Ley 34 Congreso de Colombia.,1976-11-19,NO,NaN,NaN,NaN,Activa en la Fecha de Actualizacion,2022-07-08,www.unicesar.edu.co
1,1220,UNIVERSIDAD DE ANTIOQUIA,890.980.040-8,Seccional,Departamental,Oficial,Universidad,Antioquia,Antioquia,NaN,...,8418052,No informa SIN DATO SIN DATO,3916-01-28,NO,NaN,2087,NaN,Activa en la Fecha de Actualizacion,2022-07-08,www.udea.edu.co
2,1101,UNIVERSIDAD NACIONAL DE COLOMBIA,899.999.063-3,Principal,Nacional,Oficial,Universidad,Bogotá D.C.,Bogotá D.C.,NaN,...,3165000,Ley 66 Congreso de la Rep¿blica,1899-12-30,SI,2021-08-25,15859,10,Activa en la Fecha de Actualizacion,2022-07-08,www.unal.edu.co
3,1102,UNIVERSIDAD NACIONAL DE COLOMBIA,899.999.063-3,Seccional,Nacional,Oficial,Universidad,Antioquia,Antioquia,NaN,...,4309000,Ley 66 Congreso de la Republica,1899-12-30,SI,2021-08-25,15859,10,Activa en la Fecha de Actualizacion,2022-07-08,www.medellin.unal.edu.co
4,1103,UNIVERSIDAD NACIONAL DE COLOMBIA,899.999.063-3,Seccional,Nacional,Oficial,Universidad,Caldas,Caldas,NaN,...,8879300,Ley 66 Congreso de la Republica,1899-12-30,NO,NaN,NaN,NaN,Activa en la Fecha de Actualizacion,2022-07-08,www.manizales.unal.edu.co


---
### Fin de la Parte 1
Si llegó hasta aquí sin errores, revise en su Google Drive la carpeta **EvaluadorCalidadDatosAbiertos**: debe tener el archivo `parametros.json` y, dentro de `ejecuciones`, la carpeta de esta ejecución con sus subcarpetas.

**Siguiente parte:** validación de la estructura del archivo y configuración de la evaluación (pasos 6 y 7).